# SoccerMom — YOLO 축구 모델(best.pt) 학습 (Google Colab 무료 GPU)

정밀분석 정확도의 핵심인 `best.pt`(선수/공/심판 감지 모델)를 **전부 Google 안에서** 1회 학습합니다.

- 데이터: Roboflow 무료 축구 데이터셋
- 학습: Colab 무료 GPU(T4) + Ultralytics YOLOv8
- 결과: `best.pt` → **Google Cloud Storage** 업로드 → 워커 `MODEL_PATH`에 지정

## 사용법
1. 이 파일을 https://colab.research.google.com 에 업로드
2. 상단 메뉴 **런타임 → 런타임 유형 변경 → GPU(T4)** 선택
3. 위에서부터 셀을 차례로 실행 (▶)

> 정직: 방송용 데이터로 학습한 모델이라 유소년 흔들리는 폰 영상엔 정확도가 떨어질 수 있습니다.
> 더 정확히 하려면 우리 영상 일부를 직접 라벨링해 추가 학습(파인튜닝)하면 됩니다.

## 1) GPU 확인 + 라이브러리 설치

In [ ]:
!nvidia-smi
!pip -q install ultralytics roboflow google-cloud-storage

## 2) Roboflow에서 축구 데이터셋 받기
- https://roboflow.com 무료 가입 → API 키 확인 (Settings → API)
- 공개 데이터셋: `roboflow-jvuqo/football-players-detection` (선수/공/심판/골키퍼)
- 아래 `ROBOFLOW_API_KEY`에 본인 키 입력

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "여기에_본인_Roboflow_API_키"  # @param {type:"string"}

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
dataset = project.version(1).download("yolov8")
print("데이터셋 경로:", dataset.location)

## 3) 학습 (YOLOv8)
- `epochs`: 데모는 25, 실제는 50~100 권장(시간 더 걸림)
- 무료 GPU 기준 수십 분~ 소요

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8x.pt")  # 사전학습 가중치에서 시작(전이학습)
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=25,
    imgsz=1280,
    batch=4,
    name="soccermom_best",
)
print("학습 완료. best.pt 위치: runs/detect/soccermom_best/weights/best.pt")

## 4) 간단 검증 — 샘플 이미지에 추론해보기

In [ ]:
best = YOLO("runs/detect/soccermom_best/weights/best.pt")
metrics = best.val(data=f"{dataset.location}/data.yaml")
print("mAP50:", metrics.box.map50)  # 0~1, 높을수록 좋음
print("클래스:", best.names)  # 예: {0:'ball',1:'goalkeeper',2:'player',3:'referee'}

## 5) best.pt → Google Cloud Storage 업로드
- 워커가 시작 시 GCS에서 모델을 내려받습니다 (`MODEL_PATH=gs://버킷/models/best.pt`).
- 아래 `GCP_PROJECT`, `BUCKET` 입력 후, Colab에서 Google 계정 인증.

In [ ]:
from google.colab import auth
auth.authenticate_user()  # 본인 Google 계정으로 인증(팝업)

GCP_PROJECT = "여기에_GCP_프로젝트_ID"  # @param {type:"string"}
BUCKET = "여기에_버킷이름"            # @param {type:"string"}

from google.cloud import storage
client = storage.Client(project=GCP_PROJECT)
blob = client.bucket(BUCKET).blob("models/best.pt")
blob.upload_from_filename("runs/detect/soccermom_best/weights/best.pt")
print(f"업로드 완료 → gs://{BUCKET}/models/best.pt")
print("워커 배포 시 MODEL_PATH 에 이 경로를 넣으세요.")

## 끝
이제 `soccer-analysis-worker/deploy/deploy.sh` 의 `MODEL_GS` 가 위 경로를 가리키면
정밀분석 워커가 학습된 모델로 동작합니다.

클래스 이름이 위 4)에서 본 것과 다르면, 워커 환경변수
`CLASS_PLAYER/CLASS_BALL/CLASS_REFEREE/CLASS_GOALKEEPER` 로 매핑을 맞추세요.